In [26]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict

In [27]:
load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-7B-Instruct',
    task='text-generation'
)

model = ChatHuggingFace(llm=llm)

In [28]:
class Blog(TypedDict):
    
    title : str
    outline : str
    blog : str

In [29]:
def outline_blog(state: Blog) -> Blog:
    
    title = state['title']
    
    prompt = f'Make a outline for the {title}'
    
    outline_llm = model.invoke(prompt).content
    
    state['outline'] = outline_llm
    
    return state

In [30]:
def blog_blog(state: Blog) -> Blog:
    
    blog_outcome = state['outline']
    
    prompt = f'Make a detailed blog on the topic {blog_outcome}'
    
    blog_llm = model.invoke(prompt).content
    
    state['blog'] = blog_llm
    
    return state

In [32]:
state = StateGraph(Blog)

state.add_node('outline', outline_blog)
state.add_node('blog', blog_blog)

state.add_edge(START, 'outline')
state.add_edge('outline', 'blog')
state.add_edge('blog', END)

workflow = state.compile()

In [ ]:
initial_state = {'title' : 'Technologies'}

final_workflow = workflow.invoke(initial_state)

print(final_workflow['title'])
print(final_workflow['outline'])
print(final_workflow['blog'])


{'title': 'Technologies', 'outline': 'Certainly! Here is a comprehensive outline for a topic on "Technologies":\n\n### I. Introduction\n   - A. Definition of Technology\n   - B. Importance of Technology in Modern Society\n   - C. Overview of Key Technological Categories\n\n### II. Information Technology (IT)\n   - A. Definition and Scope\n   - B. Subcategories\n      - i. Hardware (Desktops, Laptops, Servers, etc.)\n      - ii. Software (Operating Systems, Applications, Programming Languages, etc.)\n      - iii. Networking (Wired and Wireless Networks)\n   - C. Applications in Business and Daily Life\n   - D. Future Trends in IT\n\n### III. Internet of Things (IoT)\n   - A. Definition and Explanation\n   - B. Components and Architecture\n   - C. Applications in Smart Homes, Cities, and Industries\n   - D. Security and Privacy Concerns\n   - E. Future Developments\n\n### IV. Artificial Intelligence (AI) and Machine Learning (ML)\n   - A. Definitions and Fundamentals\n   - B. Application